In [34]:
import time
import numpy as np
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains

In [30]:
driver = webdriver.Chrome()
driver.maximize_window()

wait = WebDriverWait(driver, 5)

def wait_for_page_to_load(driver, wait, old_url = None):
    # To check the URL change
    if old_url:
        try:
            wait.until(EC.url_changes(old_url))
        except:
            print("The URL doesn't change within timout duration")

    # To check the loading state of the webpage
    try:
        wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
    except:
        print(f"The page \"{driver.title}\" didn't load fully within the given timeout duration")
    else:
        print(f"Successfully moved to : {driver.title}")
        print(f"Current page url: {driver.current_url}")

url = "https://finance.yahoo.com/"
driver.get(url)
wait_for_page_to_load(driver, wait, old_url = driver.current_url)

# Implement ActionChains
actions = ActionChains(driver)

# To check the market menu 
market_menu = wait.until(EC.presence_of_element_located((By.XPATH, "/html[1]/body[1]/div[1]/header[1]/div[1]/nav[1]/ol[1]/li[3]/a[1]/div[1]")))
actions.move_to_element(market_menu).perform()

# To check the stocks and most active
stocks_element = wait.until(EC.presence_of_element_located((By.XPATH, "/html[1]/body[1]/div[1]/header[1]/div[1]/nav[1]/ol[1]/li[3]/ol[1]/li[1]/a[1]/span[1]")))
actions.move_to_element(stocks_element).perform()

most_active = wait.until(EC.element_to_be_clickable((By.XPATH, "/html[1]/body[1]/div[1]/header[1]/div[1]/nav[1]/ol[1]/li[3]/ol[1]/li[1]/ol[1]/li[1]/a[1]")))
actions.move_to_element(most_active).perform()
most_active.click()

wait_for_page_to_load(driver, wait, old_url = driver.current_url)

data = []
# scraping data
while True:
    #scraping
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "table")))
    rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
    for row in rows:
        values = row.find_elements(By.TAG_NAME, "td")
        stocks = {
            "name": values[1].text,
            "symbol": values[0].text,
            "price": values[3].text,
            "change": values[4].text,
            "volume": values[6].text,
            "market_cap": values[8].text,
            "pe_ratio": values[9].text,
        }
        data.append(stocks)
    # visiting all stock pages
    try:
        next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="main-content-wrapper"]/section[1]/div/div[4]/div[3]/button[3]')))
    except:
        print("The \"next\" button is not clickable. We have navigated through all the pages")
        break
    else:
        next_button.click()
        time.sleep(1)

driver.quit()

The URL doesn't change within timout duration
Successfully moved to : Yahoo Finance - Stock Market Live, Quotes, Business & Finance News
Current page url: https://finance.yahoo.com/
Successfully moved to : Most Active Stocks: US stocks with the highest trading volume today - Yahoo Finance
Current page url: https://finance.yahoo.com/markets/stocks/most-active/
The "next" button is not clickable. We have navigated through all the pages


In [31]:
data

[{'name': 'Intel Corporation',
  'symbol': 'INTC',
  'price': '52.91',
  'change': '+2.13',
  'volume': '124.798M',
  'market_cap': '265.662B',
  'pe_ratio': '--'},
 {'name': 'NVIDIA Corporation',
  'symbol': 'NVDA',
  'price': '178.10',
  'change': '+0.46',
  'volume': '111.871M',
  'market_cap': '4.329T',
  'pe_ratio': '36.20'},
 {'name': 'Hologic, Inc.',
  'symbol': 'HOLX',
  'price': '76.01',
  'change': '0.00',
  'volume': '101.956M',
  'market_cap': '16.969B',
  'pe_ratio': '31.48'},
 {'name': 'Nokia Oyj',
  'symbol': 'NOK',
  'price': '8.85',
  'change': '-0.04',
  'volume': '76.755M',
  'market_cap': '49.405B',
  'pe_ratio': '66.82'},
 {'name': 'Plug Power Inc.',
  'symbol': 'PLUG',
  'price': '2.52',
  'change': '-0.17',
  'volume': '76.341M',
  'market_cap': '3.513B',
  'pe_ratio': '--'},
 {'name': 'Tesla, Inc.',
  'symbol': 'TSLA',
  'price': '346.65',
  'change': '-6.17',
  'volume': '65.525M',
  'market_cap': '1.301T',
  'pe_ratio': '333.88'},
 {'name': 'Grab Holdings Limi

In [33]:
len(data)

279

In [78]:
stocks_df = (
    pd
    .DataFrame(data)
    .apply(lambda col: col.str.strip() if (col.dtype == "str") else col)
    .assign(
        price = lambda df_: pd.to_numeric(df_.price),
        change = lambda df_: pd.to_numeric(df_.change.str.replace("+", "")),
        volume = lambda df_: pd.to_numeric(df_.volume.str.replace("M", "")),
        market_cap = lambda df_: df_.market_cap.apply(lambda val: float(val.replace("B", "")) if "B" in val else float(val.replace("T", "")) * 1000),
        pe_ratio = lambda df_: (
            df_
            .pe_ratio
            .replace("--", np.nan)
            .str.replace(",", "")
            .pipe(lambda col: pd.to_numeric(col))
        )
    )
    .rename(
        columns = {
            "price": "price_usd",
            "volume": "volume_M",
            "market_cap": "market_cap_B"
        }
    )
)

# stocks_df
# stocks_df.dtypes
# stocks_df.price.str.extract(r"([^0-9.])", expand = False).unique()
# stocks_df.dtypes
# stocks_df.change.str.extract(r"([^0-9.])", expand = False).unique()
# stocks_df.dtypes
# stocks_df.volume.str.extract(r"([^0-9.])", expand = False).unique()
# stocks_df.dtypes
# stocks_df.market_cap.str[-1].unique()
# stocks_df.market_cap.str.extract(r"([^0-9.])", expand = False).unique()
# stocks_df.dtypes
# stocks_df.pe_ratio.str.extract(r"([^0-9.])", expand = False).unique()
stocks_df.dtypes
stocks_df.head()

,name,symbol,price_usd,change,volume_M,market_cap_B,pe_ratio
0,Intel Corporation,INTC,52.91,2.13,124.798,265.662,NaN
1,NVIDIA Corporation,NVDA,178.10,0.46,111.871,4329.000,36.20
2,"Hologic, Inc.",HOLX,76.01,0.00,101.956,16.969,31.48
3,Nokia Oyj,NOK,8.85,-0.04,76.755,49.405,66.82
4,Plug Power Inc.,PLUG,2.52,-0.17,76.341,3.513,NaN


In [79]:
# import openpyxl
# print(openpyxl.__version__)

3.1.5


In [80]:
# stocks_df.to_excel("dummy-yahoo-stocks.xlsx", index = False)

# Restructuring COde

In [15]:
import time
import numpy as np
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains

In [19]:
class Stocksscraper:
    def __init__(self, driver, timeout = 10):
        self.driver = driver
        self.wait = WebDriverWait(self.driver, timeout)
        self.data = []

    
    def wait_for_page_to_load(self, old_url = None):
        # To check the URL change
        if old_url:
            try:
                self.wait.until(EC.url_changes(old_url))
            except:
                print("The URL doesn't change within timout duration")
    
        # To check the loading state of the webpage
        try:
            self.wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
        except:
            print(f"The page \"{self.driver.title}\" didn't load fully within the given timeout duration")
        else:
            print(f"Successfully moved to : {self.driver.title}")
            print(f"Current page url: {self.driver.current_url}")

        
    def access_url(self, url):
        self.driver.get(url)
        self.wait_for_page_to_load(old_url = self.driver.current_url)

    def access_most_active_stocks(self):
        # Implement ActionChains
        actions = ActionChains(self.driver)
        
        # To check the market menu 
        market_menu = self.wait.until(EC.presence_of_element_located((By.XPATH, "/html[1]/body[1]/div[1]/header[1]/div[1]/nav[1]/ol[1]/li[3]/a[1]/div[1]")))
        actions.move_to_element(market_menu).perform()
        
        # To check the stocks and most active
        stocks_element = self.wait.until(EC.presence_of_element_located((By.XPATH, "/html[1]/body[1]/div[1]/header[1]/div[1]/nav[1]/ol[1]/li[3]/ol[1]/li[1]/a[1]/span[1]")))
        actions.move_to_element(stocks_element).perform()
        
        most_active = self.wait.until(EC.element_to_be_clickable((By.XPATH, "/html[1]/body[1]/div[1]/header[1]/div[1]/nav[1]/ol[1]/li[3]/ol[1]/li[1]/ol[1]/li[1]/a[1]")))
        actions.move_to_element(most_active).perform()
        most_active.click()
        
        self.wait_for_page_to_load(old_url = driver.current_url)

    def extract_stocks_data(self):
        # scraping data
        while True:
            #scraping
            self.wait.until(EC.presence_of_element_located((By.TAG_NAME, "table")))
            rows = self.driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
            for row in rows:
                values = row.find_elements(By.TAG_NAME, "td")
                stocks = {
                    "name": values[1].text,
                    "symbol": values[0].text,
                    "price": values[3].text,
                    "change": values[4].text,
                    "volume": values[6].text,
                    "market_cap": values[8].text,
                    "pe_ratio": values[9].text,
                }
                self.data.append(stocks)
            # visiting all stock pages
            try:
                next_button = self.wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="main-content-wrapper"]/section[1]/div/div[4]/div[3]/button[3]')))
            except:
                print("The \"next\" button is not clickable. We have navigated through all the pages")
                break
            else:
                next_button.click()
                time.sleep(1)

    def clean_and_save_data(self, filename = "temp"):
        stocks_df = (
                pd
                .DataFrame(self.data)
                .apply(lambda col: col.str.strip() if (col.dtype == "str") else col)
                .assign(
                    price = lambda df_: pd.to_numeric(df_.price),
                    change = lambda df_: pd.to_numeric(df_.change.str.replace("+", "")),
                    volume = lambda df_: pd.to_numeric(df_.volume.str.replace("M", "")),
                    market_cap = lambda df_: df_.market_cap.apply(lambda val: float(val.replace("B", "")) if "B" in val else float(val.replace("T", "")) * 1000),
                    pe_ratio = lambda df_: (
                        df_
                        .pe_ratio
                        .replace("--", np.nan)
                        .str.replace(",", "")
                        .pipe(lambda col: pd.to_numeric(col))
                    )
                )
                .rename(
                    columns = {
                        "price": "price_usd",
                        "volume": "volume_M",
                        "market_cap": "market_cap_B"
                    }
                )
            )
        stocks_df.to_csv(f"{filename}.csv", index = False)

In [20]:
if __name__ == "__main__":
    driver = webdriver.Chrome()
    driver.maximize_window()

    url = "https://finance.yahoo.com/"
    scraper = Stocksscraper(driver, 5)

    scraper.access_url(url)
    scraper.access_most_active_stocks()
    scraper.extract_stocks_data()
    scraper.clean_and_save_data("dummy-yahoo-finanace-stocks")

    driver.quit()

The URL doesn't change within timout duration
Successfully moved to : Yahoo Finance - Stock Market Live, Quotes, Business & Finance News
Current page url: https://finance.yahoo.com/
Successfully moved to : Most Active Stocks: US stocks with the highest trading volume today - Yahoo Finance
Current page url: https://finance.yahoo.com/markets/stocks/most-active/
The "next" button is not clickable. We have navigated through all the pages
